# LeetCode 2049 — Count Nodes With the Highest Score bằng Python

Link đề: https://leetcode.com/problems/count-nodes-with-the-highest-score/ · Độ khó: **Medium** · Chủ đề: Array, Tree, Depth-First Search, Binary Tree

Skeleton LeetCode:

```python
class Solution:
    def countHighestScoreNodes(self, parents: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Biểu diễn cây từ mảng `parents` thành **danh sách con**.
2. Nhận ra: xóa nút `u` tách cây thành **các cây con của `u`** và **phần còn lại phía trên** có kích thước `n − size[u]` ⇒ điểm `= Π size[con] × (n − size[u])` (bỏ thừa số `0` khi `u` là gốc).
3. Tính **kích thước cây con** của mọi nút trong một lượt `O(n)` bằng **hậu thứ tự lặp** (tránh đệ quy sâu khi cây là một chuỗi `10⁵` nút).
4. Đếm số nút đạt **giá trị lớn nhất** trong một lượt.
5. Nhận thức về độ lớn: điểm tới `~(n/3)³ ≈ 3.7·10¹³` — cần số nguyên 64-bit ở ngôn ngữ khác.

## 2. Đề bài & ví dụ

Cho cây nhị phân gốc `0` gồm `n` nút `0..n−1`, mô tả bởi `parents` (`parents[i]` là cha của `i`, `parents[0] = −1`).

**Điểm** của một nút: xóa nút đó cùng các cạnh nối với nó, cây tách thành một hoặc nhiều cây con khác rỗng; điểm là **tích kích thước** các cây con đó.

Trả về **số nút có điểm cao nhất**.

**Ví dụ 1:** `parents = [-1,2,0,2,0]` → `3`. Điểm: nút 0: `3·1 = 3`; nút 1: `4`; nút 2: `1·1·2 = 2`; nút 3: `4`; nút 4: `4` ⇒ ba nút đạt 4.

**Ví dụ 2:** `parents = [-1,2,0]` → `2`. Điểm: nút 0: `2`; nút 1: `2`; nút 2: `1·1 = 1`.

**Ràng buộc:** `2 ≤ n ≤ 10⁵`; `parents[0] = −1`; `parents` mô tả một cây nhị phân hợp lệ.

In [1]:
import random
import sys
import time
from collections import deque
from typing import List

examples = [([-1, 2, 0, 2, 0], 3), ([-1, 2, 0], 2)]
for p, exp in examples:
    print(f"parents={p} -> kỳ vọng {exp}")

parents=[-1, 2, 0, 2, 0] -> kỳ vọng 3
parents=[-1, 2, 0] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1 Từ `parents` sang danh sách con

In [2]:
def children_of(parents):
    ch = [[] for _ in parents]
    for v, p in enumerate(parents):
        if p != -1:
            ch[p].append(v)
    return ch

print(children_of([-1, 2, 0, 2, 0]))

[[2, 4], [], [1, 3], [], []]


`parents[i]` **không** nhất thiết nhỏ hơn `i` (ở Ví dụ 1, cha của nút 1 là nút 2). Không được giả định thứ tự chỉ số là thứ tự cha–con.

### 3.2 Kích thước cây con

`size[u] = 1 + Σ size[c]` qua các con `c`. Cần tính con **trước** cha: hậu thứ tự (post-order), hoặc xử lý các nút theo thứ tự BFS **ngược**.

### 3.3 Duyệt BFS và thứ tự ngược

BFS từ gốc cho thứ tự mà cha luôn đứng trước con; đảo ngược thứ tự đó ⇒ con luôn đứng trước cha. Không cần đệ quy.

In [3]:
def bfs_order(ch):
    order, q = [], deque([0])
    while q:
        u = q.popleft(); order.append(u); q.extend(ch[u])
    return order

ch = children_of([-1, 2, 0, 2, 0])
order = bfs_order(ch); size = [1] * len(ch)
for u in reversed(order):
    for c in ch[u]: size[u] += size[c]
print("BFS:", order, "| size:", size)

BFS: [0, 2, 4, 1, 3] | size: [5, 1, 3, 1, 1]


### 3.4 Giới hạn đệ quy của Python

Mặc định khoảng 1000 tầng. Cây "chuỗi" `0 → 1 → 2 → … → 10⁵ − 1` có độ sâu `10⁵` ⇒ DFS đệ quy gây `RecursionError` (hoặc crash nếu nâng giới hạn quá cao). Lời giải an toàn dùng vòng lặp.

In [4]:
print("giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

giới hạn đệ quy hiện tại: 1000


## 4. Ý tưởng nền tảng / trực giác

Xóa nút `u`:
- Mỗi con `c` của `u` trở thành gốc một cây con kích thước `size[c]`.
- Nếu `u` không phải gốc, phần còn lại (cha và mọi thứ ngoài cây con của `u`) có `n − size[u]` nút.

⇒ `score(u) = (Π_{c con của u} size[c]) × (n − size[u] nếu u ≠ gốc)`.

Tính tay **Ví dụ 1** (`n = 5`, con: `0 → {2, 4}`, `2 → {1, 3}`):

| nút | con | size | các thành phần | điểm |
|---|---|---|---|---|
| 0 | 2, 4 | 5 | `size[2]=3`, `size[4]=1` | 3 |
| 1 | — | 1 | `5 − 1 = 4` | **4** |
| 2 | 1, 3 | 3 | `1`, `1`, `5 − 3 = 2` | 2 |
| 3 | — | 1 | `4` | **4** |
| 4 | — | 1 | `4` | **4** |

In [5]:
n = 5
for u in range(n):
    parts = [size[c] for c in ch[u]] + ([n - size[u]] if u != 0 else [])
    prod = 1
    for x in parts: prod *= x
    print(f"nút {u}: thành phần {parts} -> điểm {prod}")

nút 0: thành phần [3, 1] -> điểm 3
nút 1: thành phần [4] -> điểm 4
nút 2: thành phần [1, 1, 2] -> điểm 2
nút 3: thành phần [4] -> điểm 4
nút 4: thành phần [4] -> điểm 4


## 5. Cách 1 — Brute Force / baseline: xóa từng nút, đếm thành phần liên thông

Dựng đồ thị vô hướng; với mỗi `u`, BFS trên đồ thị bỏ `u` để lấy kích thước mọi thành phần.

In [6]:
def highest_bruteforce(parents):
    n = len(parents); g = [[] for _ in range(n)]
    for v, p in enumerate(parents):
        if p != -1:
            g[v].append(p); g[p].append(v)
    scores = []
    for u in range(n):
        seen = [False] * n; seen[u] = True; prod = 1
        for s in range(n):
            if not seen[s]:
                seen[s] = True; q = [s]; cnt = 0
                while q:
                    x = q.pop(); cnt += 1
                    for y in g[x]:
                        if not seen[y]:
                            seen[y] = True; q.append(y)
                prod *= cnt
        scores.append(prod)
    best = max(scores)
    return scores.count(best)

for p, exp in examples:
    got = highest_bruteforce(p); print(got, "kỳ vọng", exp); assert got == exp

3 kỳ vọng 3
2 kỳ vọng 2


**Độ phức tạp:** `O(n²)` — `10¹⁰` với `n = 10⁵` ⇒ TLE. Đúng theo định nghĩa nên dùng làm chuẩn.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (công thức điểm):** các thành phần sau khi xóa `u` được biết hết từ `size[·]`: cây con của từng con, cộng phần trên `n − size[u]`.

**Insight 2 (một lượt):** tính `size` cho mọi nút theo thứ tự con-trước-cha (`O(n)`), rồi tính điểm từng nút `O(số con) = O(1)` (cây nhị phân).

**Insight 3 (đếm max):** duy trì `best` và `cnt`: điểm lớn hơn ⇒ `best = điểm, cnt = 1`; bằng ⇒ `cnt += 1`.

**Lưu ý gốc:** với gốc, `n − size[0] = 0` — **không** nhân vào (phần trên rỗng không phải cây con).

**Invariant (hậu thứ tự):** khi xử lý `u`, mọi con của `u` đã có `size` đúng.

### 6.1 Minh họa: nút "trung tâm" có điểm cao

Điểm là tích các phần có tổng `n − 1` ⇒ lớn nhất khi các phần **cân bằng** (bất đẳng thức AM–GM). In điểm mọi nút của một cây nhị phân hoàn chỉnh 15 nút và một chuỗi 7 nút.

In [7]:
def scores_of(parents):
    ch = children_of(parents); n = len(parents); order = bfs_order(ch); size = [1] * n
    for u in reversed(order):
        for c in ch[u]: size[u] += size[c]
    res = []
    for u in range(n):
        s = n - size[u] if u else 1
        for c in ch[u]: s *= size[c]
        res.append(s)
    return res

full = [-1] + [(i - 1) // 2 for i in range(1, 15)]
chain = [-1] + list(range(6))
print("cây hoàn chỉnh 15 nút:", scores_of(full))
print("chuỗi 7 nút:", scores_of(chain), "-> nút giữa chuỗi điểm 3·3 = 9 cao nhất")

cây hoàn chỉnh 15 nút: [49, 72, 72, 12, 12, 12, 12, 14, 14, 14, 14, 14, 14, 14, 14]
chuỗi 7 nút: [6, 5, 8, 9, 8, 5, 6] -> nút giữa chuỗi điểm 3·3 = 9 cao nhất


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Size theo thứ tự BFS ngược (bản nộp)

In [8]:
def highest_bfs(parents):
    n = len(parents); ch = children_of(parents)
    order = bfs_order(ch); size = [1] * n
    for u in reversed(order):
        for c in ch[u]:
            size[u] += size[c]
    best = cnt = 0
    for u in range(n):
        s = n - size[u] if u != 0 else 1
        for c in ch[u]:
            s *= size[c]
        if s > best:
            best, cnt = s, 1
        elif s == best:
            cnt += 1
    return cnt

print([highest_bfs(p) for p, _ in examples])

[3, 2]


**Độ phức tạp Cách 2:** `O(n)` thời gian và bộ nhớ.

### 7.2 Cách 3 — DFS đệ quy (cần nâng giới hạn đệ quy)

Viết tự nhiên nhất: `dfs(u)` trả `size[u]` và tính điểm ngay. Ổn với cây cân bằng; với chuỗi dài phải `sys.setrecursionlimit` và vẫn có nguy cơ tràn stack C.

In [9]:
def highest_recursive(parents):
    n = len(parents); ch = children_of(parents)
    best = cnt = 0
    def dfs(u):
        nonlocal best, cnt
        s, total = 1, 1
        for c in ch[u]:
            sz = dfs(c); s *= sz; total += sz
        if u != 0:
            s *= n - total
        if s > best: best, cnt = s, 1
        elif s == best: cnt += 1
        return total
    dfs(0)
    return cnt

print([highest_recursive(p) for p, _ in examples])

[3, 2]


### 7.3 Cách 4 — Bóc lá từ dưới lên (đếm số con chưa xử lý)

Như thuật toán Kahn trên cây: hàng đợi ban đầu là các lá; khi xử lý `u`, cộng `size[u]` vào cha và giảm "số con chưa xong" của cha; cha về 0 thì vào hàng đợi. Không cần danh sách con cho bước tính `size` (chỉ cần `parents`), nhưng vẫn cần biết kích thước từng con để tính điểm — lưu tích `prodc[p] *= size[u]` khi đẩy lên.

In [10]:
class Solution:
    def countHighestScoreNodes(self, parents: List[int]) -> int:
        n = len(parents)
        pending = [0] * n
        for v in range(1, n):
            pending[parents[v]] += 1
        size = [1] * n
        prodc = [1] * n                         # tích kích thước các cây con
        q = deque(v for v in range(n) if pending[v] == 0)
        best = cnt = 0
        while q:
            u = q.popleft()
            s = prodc[u] * (n - size[u] if u != 0 else 1)
            if s > best:
                best, cnt = s, 1
            elif s == best:
                cnt += 1
            p = parents[u]
            if p != -1:
                size[p] += size[u]
                prodc[p] *= size[u]
                pending[p] -= 1
                if pending[p] == 0:
                    q.append(p)
        return cnt

print([Solution().countHighestScoreNodes(p) for p, _ in examples])

[3, 2]


Khi `u` ra khỏi hàng đợi, mọi con của nó đã xử lý xong ⇒ `size[u]` và `prodc[u]` đầy đủ.

**Độ phức tạp Cách 4:** `O(n)`.

## 8. TRACE từng bước

Trace Cách 4 trên **Ví dụ 1** `[-1,2,0,2,0]`:

In [11]:
def trace(parents):
    n = len(parents); pending = [0] * n
    for v in range(1, n): pending[parents[v]] += 1
    size, prodc = [1] * n, [1] * n
    q = deque(v for v in range(n) if pending[v] == 0); best = cnt = 0
    print("lá ban đầu:", list(q))
    while q:
        u = q.popleft(); s = prodc[u] * (n - size[u] if u else 1)
        if s > best: best, cnt = s, 1
        elif s == best: cnt += 1
        print(f"xử lý {u}: size={size[u]}, tích con={prodc[u]}, phần trên={n - size[u] if u else '-'} -> điểm {s}; best={best}, cnt={cnt}")
        p = parents[u]
        if p != -1:
            size[p] += size[u]; prodc[p] *= size[u]; pending[p] -= 1
            if pending[p] == 0: q.append(p)
    return cnt

assert trace([-1, 2, 0, 2, 0]) == 3

lá ban đầu: [1, 3, 4]
xử lý 1: size=1, tích con=1, phần trên=4 -> điểm 4; best=4, cnt=1
xử lý 3: size=1, tích con=1, phần trên=4 -> điểm 4; best=4, cnt=2
xử lý 4: size=1, tích con=1, phần trên=4 -> điểm 4; best=4, cnt=3
xử lý 2: size=3, tích con=1, phần trên=2 -> điểm 2; best=4, cnt=3
xử lý 0: size=5, tích con=3, phần trên=- -> điểm 3; best=4, cnt=3


| thứ tự | nút | size | tích con | phần trên | điểm | best, cnt |
|---|---|---|---|---|---|---|
| 1 | 1 | 1 | 1 | 4 | 4 | 4, 1 |
| 2 | 3 | 1 | 1 | 4 | 4 | 4, 2 |
| 3 | 4 | 1 | 1 | 4 | 4 | 4, 3 |
| 4 | 2 | 3 | 1·1 | 2 | 2 | 4, 3 |
| 5 | 0 | 5 | 3·1 | — | 3 | 4, 3 |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 4** — bóc lá từ dưới lên, hoàn toàn lặp, `O(n)`.

In [12]:
class Solution:
    def countHighestScoreNodes(self, parents: List[int]) -> int:
        n = len(parents)
        pending = [0] * n
        for v in range(1, n):
            pending[parents[v]] += 1
        size = [1] * n
        prodc = [1] * n
        q = deque(v for v in range(n) if pending[v] == 0)
        best = cnt = 0
        while q:
            u = q.popleft()
            s = prodc[u] * (n - size[u] if u != 0 else 1)
            if s > best:
                best, cnt = s, 1
            elif s == best:
                cnt += 1
            p = parents[u]
            if p != -1:
                size[p] += size[u]
                prodc[p] *= size[u]
                pending[p] -= 1
                if pending[p] == 0:
                    q.append(p)
        return cnt

for p, exp in examples:
    got = Solution().countHighestScoreNodes(p); print(got, exp); assert got == exp

3 3
2 2


## 10. Vì sao thuật toán đúng

**Thành phần sau khi xóa `u`.** Trong cây, xóa `u` và các cạnh kề: mỗi cạnh `(u, c)` với con `c` tách cây con gốc `c` (kích thước `size[c]`) — các cây con này rời nhau và không nối với phần khác vì cây không có chu trình. Nếu `u ≠ 0`, cạnh `(u, cha)` tách phần còn lại: mọi nút **không** thuộc cây con của `u`, kích thước `n − size[u] ≥ 1`, liên thông qua cha. Không còn thành phần nào khác. Vậy điểm đúng như công thức.

**Tính `size`.** Trong Cách 4, một nút vào hàng đợi khi mọi con đã xử lý (`pending = 0`); quy nạp theo chiều cao: lá có `size = 1`, `prodc = 1` (tích rỗng). Khi con `u` xử lý xong, ta cộng `size[u]` vào cha và nhân vào `prodc[cha]` ⇒ khi cha ra khỏi hàng đợi, `size` và `prodc` đầy đủ. Mọi nút đều được xử lý đúng một lần (cây liên thông, gốc là nút duy nhất không có cha).

**Đếm max.** Bất biến: `best` là điểm lớn nhất trong các nút đã xử lý, `cnt` là số nút đạt nó. ∎

### 10.1 Kiểm chứng thực nghiệm

Với cây nhị phân ngẫu nhiên (đánh số **xáo trộn** để cha không nhất thiết có chỉ số nhỏ hơn), so điểm của từng nút giữa công thức và cách xóa–BFS, và kiểm tra `Σ thành phần = n − 1`.

In [13]:
def rand_binary_tree(rng, n):
    # sinh cây nhị phân ngẫu nhiên rồi đổi nhãn sao cho gốc vẫn là 0
    par = [-1] * n; slots = [0, 0]
    for v in range(1, n):
        i = rng.randrange(len(slots)); p = slots[i]; slots[i] = slots[-1]; slots.pop()
        par[v] = p; slots += [v, v]
    perm = [0] + rng.sample(range(1, n), n - 1)
    out = [-1] * n
    for v in range(1, n):
        out[perm[v]] = perm[par[v]]
    return out

def brute_scores(parents):
    n = len(parents); g = [[] for _ in range(n)]
    for v, p in enumerate(parents):
        if p != -1: g[v].append(p); g[p].append(v)
    res = []
    for u in range(n):
        seen = [False] * n; seen[u] = True; prod = 1; parts = []
        for s in range(n):
            if not seen[s]:
                seen[s] = True; st = [s]; c = 0
                while st:
                    x = st.pop(); c += 1
                    for y in g[x]:
                        if not seen[y]: seen[y] = True; st.append(y)
                parts.append(c); prod *= c
        assert sum(parts) == n - 1
        res.append(prod)
    return res

rng = random.Random(1010)
for _ in range(300):
    p = rand_binary_tree(rng, rng.randint(2, 25))
    assert all(len(c) <= 2 for c in children_of(p))
    assert scores_of(p) == brute_scores(p)
print("Điểm theo công thức = điểm theo định nghĩa (300 cây, nhãn xáo trộn)")

Điểm theo công thức = điểm theo định nghĩa (300 cây, nhãn xáo trộn)


## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Xóa từng nút + BFS | `O(n²)` | `O(n)` |
| 2. BFS ngược | `O(n)` | `O(n)` |
| 3. DFS đệ quy | `O(n)` | `O(h)` stack (nguy hiểm khi `h` lớn) |
| 4. Bóc lá | `O(n)` | `O(n)` |

In [14]:
rng = random.Random(2049)
big = rand_binary_tree(rng, 10 ** 5)
chain_big = [-1] + list(range(10 ** 5 - 1))
for name, f in [("bfs", highest_bfs), ("submit", Solution().countHighestScoreNodes)]:
    t = time.perf_counter(); r1 = f(big); r2 = f(chain_big)
    print(f"{name:<7}: cây ngẫu nhiên {r1}, chuỗi {r2} ({(time.perf_counter() - t) * 1000:.0f} ms)")
small = rand_binary_tree(rng, 1500)
t = time.perf_counter(); highest_bruteforce(small); print(f"brute n = 1500: {(time.perf_counter() - t) * 1000:.0f} ms")

bfs    : cây ngẫu nhiên 1, chuỗi 2 (139 ms)
submit : cây ngẫu nhiên 1, chuỗi 2 (68 ms)


brute n = 1500: 258 ms


## 12. Bộ test

In [15]:
ALL = {"bfs": highest_bfs, "recursive": highest_recursive, "submit": Solution().countHighestScoreNodes}
def run_all(p, brute=True):
    res = {k: f(p) for k, f in ALL.items()}
    if brute:
        res["brute"] = highest_bruteforce(p)
    assert len(set(res.values())) == 1, (p, res)
    return res["submit"]

for p, exp in examples:
    assert run_all(p) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm bằng vét cạn trong run_all
edge = [([-1, 0], 2), ([-1, 0, 0], 2), ([-1, 0, 1], 2), ([-1, 0, 1, 2], 2), ([-1, 0, 0, 1, 1, 2, 2], 1), ([-1, 2, 0], 2)]
for p, exp in edge:
    assert run_all(p) == exp, (p, run_all(p))
rng = random.Random(20490)
for _ in range(400):
    run_all(rand_binary_tree(rng, rng.randint(2, 30)))
for _ in range(5):
    run_all(rand_binary_tree(rng, 3000), brute=False)
print("Ví dụ, 6 edge case, 400 cây nhỏ (đối chiếu vét cạn), 5 cây 3000 nút: OK")

Ví dụ, 6 edge case, 400 cây nhỏ (đối chiếu vét cạn), 5 cây 3000 nút: OK


### 12.1 Kiểm tra tính chất

- Đổi nhãn các nút (giữ gốc 0) không đổi đáp án.
- Tổng các thành phần của mỗi nút là `n − 1`; điểm mỗi nút `≥ 1`.
- Đáp án `≥ 1` và `≤ n`; với chuỗi `n ≥ 6` nút, `n` lẻ thì chỉ nút giữa đạt max (đáp án 1), `n` chẵn thì hai nút giữa (đáp án 2). (Với `n = 5`, gốc, lá và nút giữa cùng điểm 4.)

In [16]:
rng = random.Random(121)
f = Solution().countHighestScoreNodes
for _ in range(300):
    n = rng.randint(2, 40); p = rand_binary_tree(rng, n)
    perm = [0] + rng.sample(range(1, n), n - 1); q = [-1] * n
    for v in range(1, n): q[perm[v]] = perm[p[v]]
    assert f(q) == f(p) and 1 <= f(p) <= n and min(scores_of(p)) >= 1
for n in range(6, 60):
    assert f([-1] + list(range(n - 1))) == (1 if n % 2 else 2)
print("3 tính chất đúng")

3 tính chất đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nhân cả `n − size[0] = 0` cho gốc

In [17]:
def wrong_root(parents):
    n = len(parents); ch = children_of(parents); size = [1] * n
    for u in reversed(bfs_order(ch)):
        for c in ch[u]: size[u] += size[c]
    sc = []
    for u in range(n):
        s = n - size[u]                                   # SAI: gốc -> 0
        for c in ch[u]: s *= size[c]
        sc.append(s)
    return sc.count(max(sc))

for p, exp in examples:
    print("sai :", wrong_root(p), "| đúng:", exp)

sai : 3 | đúng: 3
sai : 1 | đúng: 2


**Vì sao sai:** gốc không có "phần trên"; nhân với `0` làm điểm gốc thành 0. Ở Ví dụ 2 gốc lẽ ra có điểm 2 (đạt max). **Sửa:** dùng `1` thay cho `n − size[u]` khi `u` là gốc.

### Lỗi 2 — Giả định `parents[i] < i`, cộng size theo chỉ số giảm dần

In [18]:
def wrong_index_order(parents):
    n = len(parents); size = [1] * n
    for v in range(n - 1, 0, -1):
        size[parents[v]] += size[v]                       # SAI nếu cha có chỉ số lớn hơn con
    ch = children_of(parents); sc = []
    for u in range(n):
        s = n - size[u] if u else 1
        for c in ch[u]: s *= size[c]
        sc.append(s)
    return sc.count(max(sc))

for p, exp in examples:
    print("sai :", wrong_index_order(p), "| đúng:", exp, "(trùng do may mắn)")
p = [-1, 2, 3, 0]          # tự dựng (đối chiếu lại đề gốc): chuỗi 0 -> 3 -> 2 -> 1
print("sai :", wrong_index_order(p), "| đúng:", Solution().countHighestScoreNodes(p), "| brute:", highest_bruteforce(p))

sai : 3 | đúng: 3 (trùng do may mắn)
sai : 2 | đúng: 2 (trùng do may mắn)
sai : 1 | đúng: 2 | brute: 2


**Vì sao sai:** khi cha có chỉ số **lớn hơn** con, `size[cha]` có thể đã được cộng vào ông trước khi nhận phần của con. Ở Ví dụ 1 (`parents[1] = 2`), `size[0]` chỉ ra 4 thay vì 5 nhưng đáp án tình cờ vẫn đúng; với chuỗi `0 → 3 → 2 → 1`, `size[0]` sai làm điểm gốc sai và đáp án sai. **Sửa:** xử lý theo thứ tự cây (BFS ngược / bóc lá).

### Lỗi 3 — DFS đệ quy trên cây chuỗi dài

In [19]:
chain = [-1] + list(range(4999))
try:
    highest_recursive(chain)
    print("không lỗi (giới hạn đệ quy đã được nâng?)")
except RecursionError as e:
    print("RecursionError:", str(e)[:50], "| bản lặp:", Solution().countHighestScoreNodes(chain))

RecursionError: maximum recursion depth exceeded | bản lặp: 2


**Vì sao sai:** cây nhị phân có thể suy biến thành chuỗi độ sâu `n`; Python mặc định chỉ ~1000 tầng. Nâng giới hạn (`sys.setrecursionlimit`) có thể làm tràn **stack C** và crash tiến trình. **Sửa:** duyệt lặp.

### Lỗi 4 — Tràn số 32-bit (ở C++/Java)

In [20]:
def as_int32(x):
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >= 1 << 31 else x

def wrong_int32(parents):
    sc = [as_int32(s) for s in scores_of(parents)]       # mô phỏng nhân bằng int 32-bit
    return sc.count(max(sc))

# tự dựng (đối chiếu lại đề gốc): 0 -> 1; nút 1 có hai chuỗi 30000 nút; gốc có thêm một chuỗi 30000 nút
L = 30000
p = [-1, 0] + [1] + list(range(2, 2 + L - 1)) + [1] + list(range(2 + L, 2 + 2 * L - 1)) + [0] + list(range(2 + 2 * L, 2 + 3 * L - 1))
assert all(len(c) <= 2 for c in children_of(p)) and len(p) == 3 * L + 2
sc = scores_of(p); sc32 = [as_int32(x) for x in sc]
print("điểm lớn nhất thật:", max(sc), "tại nút", sc.index(max(sc)), "| > 2^31:", max(sc) > 2 ** 31)
m32 = max(sc32)
print("nếu nhân bằng int32: điểm nút 1 thành", sc32[1], "| nút đạt max:", [u for u in range(len(p)) if sc32[u] == m32][:5], "(sai nút)")

điểm lớn nhất thật: 27000900000000 tại nút 1 | > 2^31: True
nếu nhân bằng int32: điểm nút 1 thành -1559389952 | nút đạt max: [0] (sai nút)


**Vì sao sai:** điểm có thể tới khoảng `(n/3)³` — với `n = 10⁵` là `~3.7·10¹³`, vượt xa `2³¹`. Trong Python không lỗi, nhưng nhớ dùng `long` khi viết bằng Java/C++. Cây ở trên: gốc có hai nhánh chuỗi; nút ở **giữa** một nhánh có điểm là tích ba số lớn.

### Lỗi 5 — Đặt lại `cnt = 1` cả khi **bằng** max

In [21]:
def wrong_tie(parents):
    best = cnt = 0
    for s in scores_of(parents):
        if s >= best:                                     # SAI: hòa cũng reset cnt
            best, cnt = s, 1
    return cnt

for p_, exp in examples:
    print("sai :", wrong_tie(p_), "| đúng:", exp)

sai : 1 | đúng: 3
sai : 1 | đúng: 2


**Vì sao sai:** khi điểm **bằng** `best`, phải **tăng** `cnt` chứ không đặt lại 1. Ví dụ 1 có ba nút điểm 4 nhưng bản sai trả 1. **Sửa:** tách hai nhánh `>` (đặt lại) và `==` (tăng).

### Lỗi 6 — Dùng `n − size[u] − 1` cho phần trên

In [22]:
def wrong_minus_one(parents):
    n = len(parents); ch = children_of(parents); size = [1] * n
    for u in reversed(bfs_order(ch)):
        for c in ch[u]: size[u] += size[c]
    sc = []
    for u in range(n):
        s = (n - size[u] - 1) if u else 1                 # SAI: size[u] đã gồm cả u
        for c in ch[u]: s *= size[c]
        sc.append(s)
    return sc.count(max(sc))

for p, exp in examples:
    print("sai :", wrong_minus_one(p), "| đúng:", exp)

sai : 4 | đúng: 3
sai : 1 | đúng: 2


**Vì sao sai:** `size[u]` đã tính cả `u` ⇒ phần trên là `n − size[u]`, trừ thêm 1 là đếm hụt. **Sửa:** kiểm tra với lá: phần trên của lá phải là `n − 1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xóa một nút/cạnh của cây, xét các mảnh" ⇒ **kích thước cây con** + phần bù `n − size`.
- Cây từ mảng `parents`: dựng danh sách con; **không** giả định cha có chỉ số nhỏ hơn.
- `n` tới `10⁵` và cây có thể là chuỗi ⇒ duyệt **lặp** (BFS ngược hoặc bóc lá).
- Đếm phần tử đạt max trong một lượt: `best`, `cnt`.
- Tích nhiều kích thước ⇒ kiểm tra tràn số.

## 15. Biến thể của bài

1. **Cây tổng quát** (không nhị phân) — công thức giữ nguyên, tích qua mọi con.
2. **Trọng tâm cây (centroid)**: nút làm **cực tiểu** kích thước mảnh lớn nhất — cùng `size` (Bài tập 5).
3. **2322 Minimum Score After Removals on a Tree**: xóa **hai** cạnh, XOR các mảnh.
4. **834 Sum of Distances in Tree**: "re-rooting" dùng `size` và `n − size`.
5. **Xóa cạnh** thay vì nút: tích hai mảnh `size[c]·(n − size[c])`.

### 15.1 Code biến thể: xóa một **cạnh** để tích hai mảnh lớn nhất (1339-style)

In [23]:
def max_edge_product(parents):
    n = len(parents); ch = children_of(parents); size = [1] * n
    for u in reversed(bfs_order(ch)):
        for c in ch[u]: size[u] += size[c]
    return max(size[v] * (n - size[v]) for v in range(1, n))

def max_edge_product_brute(parents):
    n = len(parents); best = 0
    for v in range(1, n):
        seen = {v}; st = [v]                              # BFS từ v trên cây bỏ cạnh (v, cha)
        g2 = [[] for _ in range(n)]
        for x, p in enumerate(parents):
            if p != -1 and x != v: g2[x].append(p); g2[p].append(x)
        for x, p in enumerate(parents):
            if p == v: g2[x].append(v); g2[v].append(x)
        while st:
            x = st.pop()
            for y in g2[x]:
                if y not in seen: seen.add(y); st.append(y)
        best = max(best, len(seen) * (n - len(seen)))
    return best

rng = random.Random(15)
for _ in range(200):
    p = rand_binary_tree(rng, rng.randint(2, 20)); assert max_edge_product(p) == max_edge_product_brute(p)
print("Xóa cạnh: công thức size·(n − size) khớp vét cạn;", max_edge_product([-1, 2, 0, 2, 0]))

Xóa cạnh: công thức size·(n − size) khớp vét cạn; 6


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Xóa + BFS | Định nghĩa | `O(n²)` | `O(n)` | Kiểm chứng |
| 2. BFS ngược | size con trước cha | `O(n)` | `O(n)` | Dễ hiểu |
| 3. DFS đệ quy | Tự nhiên | `O(n)` | `O(h)` | Cây nông |
| 4. Bóc lá | Kahn trên cây | `O(n)` | `O(n)` | **Bản nộp** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[-1,0]`; (b) `[-1,0,0]`; (c) `[-1,0,1]`; (d) `[-1,0,1,2]`.

**Bài 2 (tính tay):** Với `[-1,0,0,1,1,2,2]` (cây hoàn chỉnh 7 nút), tính `size` và điểm từng nút.

**Bài 3 (điền TODO):** Hoàn thiện `subtree_sizes(parents)` bằng BFS ngược.

**Bài 4 (tự viết lại):** Viết `all_scores(parents)` cho **cây tổng quát** (số con tùy ý), trả danh sách điểm.

**Bài 5 (mini challenge):** Viết `centroids(parents)` trả danh sách **trọng tâm** (nút cực tiểu hóa kích thước mảnh lớn nhất sau khi xóa) của cây tổng quát, `O(n)`; đối chiếu vét cạn. Kiểm tra: luôn có 1 hoặc 2 trọng tâm, và mảnh lớn nhất `≤ n/2`.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def subtree_sizes(parents):
    ch = children_of(parents); size = [1] * len(parents)
    # TODO: với u trong reversed(bfs_order(ch)): cộng size các con vào size[u]
    return size

In [25]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def all_scores(parents):
    # TODO: như scores_of nhưng không giả định tối đa 2 con
    pass

def centroids(parents):
    # TODO: với mỗi u: max(max size con, n - size[u]); lấy các u đạt cực tiểu
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: điểm lá = `n − 1`. Mức 2: gốc = tích size con. Mức 3: (a) điểm `[1, 1]` ⇒ `2`; (b) gốc `1·1 = 1`, hai lá `2` ⇒ `2`; (c) chuỗi 3: `[2, 1, 2]` ⇒ `2`; (d) chuỗi 4: `[3, 2, 2, 3]` ⇒ `2`.

**Bài 2.** Mức 1: `size = [7,3,3,1,1,1,1]`. Mức 2: gốc `3·3 = 9`; nút 1, 2: `1·1·4 = 4`; lá `6`. Mức 3: max `9` duy nhất ⇒ `1`.

**Bài 3.** Mức 1: BFS từ 0. Mức 2: duyệt ngược. Mức 3: `size[u] += size[c]`.

**Bài 4.** Mức 1: tích qua mọi con. Mức 2: thêm phần trên nếu không phải gốc. Mức 3: dùng lại `bfs_order`.

**Bài 5.** Mức 1: mảnh lớn nhất của `u` = `max(size[c] for c, n − size[u])`. Mức 2: lấy `min` qua `u`. Mức 3: định lý: luôn có trọng tâm với mảnh lớn nhất `≤ ⌊n/2⌋`, tối đa 2 trọng tâm (kề nhau).

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2, 3
f = Solution().countHighestScoreNodes
tests = [[-1, 0], [-1, 0, 0], [-1, 0, 1], [-1, 0, 1, 2]]
print("Bài 1:", [scores_of(t) for t in tests], [f(t) for t in tests]); assert [f(t) for t in tests] == [2, 2, 2, 2]
full7 = [-1, 0, 0, 1, 1, 2, 2]
print("Bài 2:", scores_of(full7), f(full7)); assert scores_of(full7) == [9, 4, 4, 6, 6, 6, 6] and f(full7) == 1
def subtree_sizes(parents):
    ch = children_of(parents); size = [1] * len(parents)
    for u in reversed(bfs_order(ch)):
        for c in ch[u]: size[u] += size[c]
    return size
assert subtree_sizes([-1, 2, 0, 2, 0]) == [5, 1, 3, 1, 1]
print("Bài 3: OK")

Bài 1: [[1, 1], [1, 2, 2], [2, 1, 2], [3, 2, 2, 3]] [2, 2, 2, 2]
Bài 2: [9, 4, 4, 6, 6, 6, 6] 1
Bài 3: OK


In [27]:
# Bài 4, 5
def rand_tree(rng, n):
    par = [-1] + [rng.randrange(v) for v in range(1, n)]
    perm = [0] + rng.sample(range(1, n), n - 1); out = [-1] * n
    for v in range(1, n): out[perm[v]] = perm[par[v]]
    return out

def all_scores(parents):
    n = len(parents); ch = children_of(parents); size = subtree_sizes(parents); res = []
    for u in range(n):
        s = n - size[u] if u else 1
        for c in ch[u]: s *= size[c]
        res.append(s)
    return res

def centroids(parents):
    n = len(parents); ch = children_of(parents); size = subtree_sizes(parents)
    worst = [max([size[c] for c in ch[u]] + ([n - size[u]] if u else [0])) for u in range(n)]
    m = min(worst)
    return [u for u in range(n) if worst[u] == m]

def centroids_brute(parents):
    n = len(parents); g = [[] for _ in range(n)]
    for v, p in enumerate(parents):
        if p != -1: g[v].append(p); g[p].append(v)
    worst = []
    for u in range(n):
        seen = [False] * n; seen[u] = True; big = 0
        for s in range(n):
            if not seen[s]:
                seen[s] = True; st = [s]; c = 0
                while st:
                    x = st.pop(); c += 1
                    for y in g[x]:
                        if not seen[y]: seen[y] = True; st.append(y)
                big = max(big, c)
        worst.append(big)
    m = min(worst); return [u for u in range(n) if worst[u] == m]

rng = random.Random(5)
for _ in range(300):
    p = rand_tree(rng, rng.randint(2, 25))
    assert all_scores(p) == brute_scores(p)
    c = centroids(p); assert c == centroids_brute(p) and 1 <= len(c) <= 2
print("Bài 4, 5: OK |", centroids([-1, 0, 1, 2, 3]), centroids([-1, 0, 1, 2]))

Bài 4, 5: OK | [2] [1, 2]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
size = [1] * n; prodc = [1] * n; pending[p] = số con
q = các lá
while q:
    u = q.popleft()
    score = prodc[u] * (n - size[u] if u else 1)
    cập nhật best / cnt
    p = parents[u]
    if p != -1:
        size[p] += size[u]; prodc[p] *= size[u]
        pending[p] -= 1; if pending[p] == 0: q.append(p)
return cnt
```

**Checklist:**
- [ ] Gốc: **không** nhân `n − size[0] = 0`?
- [ ] Phần trên là `n − size[u]` (không trừ thêm 1)?
- [ ] Không giả định `parents[i] < i`?
- [ ] Duyệt **lặp** (cây chuỗi `10⁵` nút)?
- [ ] Số lớn: 64-bit ở ngôn ngữ khác?